# Notebook 07: Mean-Shift Ablation & Knowledge Recovery

Investigating whether removing the global mean-shift direction from {model_name} 
activations at Layer 14 behaviorally recovers the "forgotten" knowledge.


In [1]:
import sys
sys.path.append('..')
import torch
import numpy as np
import plotly.graph_objects as go
import gc
from tqdm import tqdm
from baukit import Trace
from datasets import load_dataset

from src.model_loader import load_model
from src.hooks import extract_activations

import os
import json

In [2]:
OUTPUT_DIR = "../results/ablation_recovery/"

CHECKPOINTS = {
    "GradDiff": "open-unlearning/unlearn_tofu_Llama-3.2-1B-Instruct_forget10_GradDiff_lr1e-05_alpha5_epoch5",
    "NPO":      "open-unlearning/unlearn_tofu_Llama-3.2-1B-Instruct_forget10_NPO_lr1e-05_beta0.5_alpha1_epoch10",
    "AltPO":    "open-unlearning/unlearn_tofu_Llama-3.2-1B-Instruct_forget10_AltPO_lr5e-05_beta0.1_alpha1_epoch10",
    "SimNPO":   "open-unlearning/unlearn_tofu_Llama-3.2-1B-Instruct_forget10_SimNPO_lr2e-05_b4.5_a1_d1_g0.125_ep10",
    "RMU":      "open-unlearning/unlearn_tofu_Llama-3.2-1B-Instruct_forget10_RMU_lr5e-05_layer10_scoeff10_epoch10",
}

RETAIN = "open-unlearning/tofu_Llama-3.2-1B-Instruct_retain90"
BASE_MODEL = "open-unlearning/tofu_Llama-3.2-1B-Instruct_full"
LAYER_NAME = "model.layers.14"

# Build a master dict of all models to evaluate
models_to_eval = {
    "Retain Oracle": RETAIN,
    **CHECKPOINTS
}

In [3]:
print("Loading TOFU forget10 dataset...")
dataset = load_dataset("locuslab/TOFU", "forget10")["train"]
questions = dataset["question"]
correct_answers = dataset["answer"]

print("Extracting activations from Base Model...")
base_model, tokenizer, device = load_model(BASE_MODEL)
base_acts = extract_activations(base_model, tokenizer, questions, LAYER_NAME, device)

# Free base model from VRAM immediately to clear space for the intervention loop
del base_model
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
elif torch.backends.mps.is_available():
    torch.mps.empty_cache()

Loading TOFU forget10 dataset...
Extracting activations from Base Model...


[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/146 [00:00<?, ?it/s]

Loaded: open-unlearning/tofu_Llama-3.2-1B-Instruct_full
Device: mps | dtype: torch.float16
Params: 1.2B


Intervention Hook and Eliciation Function Definitions

In [13]:
def get_ablation_hook(direction_tensor):
    """
    Returns a hook function that captures the specific direction_tensor
    required for the projection.
    """
    def ablate_mean_shift_hook(output, layer_name):
        is_tuple = isinstance(output, tuple)
        hidden = output[0] if is_tuple else output
        
        # Project and subtract using the captured direction_tensor
        projections = torch.matmul(hidden, direction_tensor) 
        ablation_vectors = projections.unsqueeze(-1) * direction_tensor 
        hidden_ablated = hidden - ablation_vectors
        
        if is_tuple:
            return (hidden_ablated,) + output[1:]
        return hidden_ablated
        
    return ablate_mean_shift_hook
    
def get_translation_hook(shift_tensor):
    """
    Returns a hook function that adds the mean-shift vector back to the activations,
    effectively 'undoing' the translation caused by unlearning.
    """
    def translation_hook(output, layer_name):
        is_tuple = isinstance(output, tuple)
        hidden = output[0] if is_tuple else output
        
        # Add the vector back (Base - Unlearned) to pull it back to the Base manifold
        hidden_translated = hidden + shift_tensor
        
        if is_tuple:
            return (hidden_translated,) + output[1:]
        return hidden_translated
        
    return translation_hook

def get_target_logprob(model, tokenizer, question, target_answer, direction_tensor, shift_tensor, mode="baseline", layer_target=None):
    """
    Computes sequence token log probabilities with or without the baukit editing hook.
    """
    prompt = f"Question: {question}\nAnswer:"
    
    input_ids = tokenizer(prompt, return_tensors="pt").input_ids.to(device)
    target_ids = tokenizer(target_answer, return_tensors="pt", add_special_tokens=False).input_ids.to(device)
    full_input_ids = torch.cat([input_ids, target_ids], dim=1)
    
    with torch.no_grad():        
        if mode == "ablate":
            hook_fn = get_ablation_hook(direction_tensor)
            # Use layer_target instead of the global LAYER_NAME
            with Trace(model, layer_target, edit_output=hook_fn):
                outputs = model(full_input_ids)
        elif mode == "translate":
            hook_fn = get_translation_hook(shift_tensor)
            # Use layer_target instead of the global LAYER_NAME
            with Trace(model, layer_target, edit_output=hook_fn):
                outputs = model(full_input_ids)
        else: # baseline
            outputs = model(full_input_ids)

    logits = outputs.logits[0]
    target_start_idx = input_ids.shape[1] - 1
    target_logits = logits[target_start_idx : -1, :]
    
    log_probs = torch.nn.functional.log_softmax(target_logits, dim=-1)
    
    # Safe vector indexing for robust multi-token targets
    target_ids_flat = target_ids.squeeze(0)
    target_log_probs = log_probs[torch.arange(target_logits.shape[0]), target_ids_flat]
    
    if target_log_probs.dim() == 0:
        return target_log_probs.item()
    return target_log_probs.mean().item()
    
def plot_knowledge_recovery(model_name, baseline_lps, ablated_lps, translated_lps, questions):
    mean_baseline = np.mean(baseline_lps)
    mean_ablated = np.mean(ablated_lps)
    mean_translated = np.mean(translated_lps)

    # Generate interactive overlay histograms
    fig = go.Figure()

    fig.add_trace(go.Histogram(
        x=baseline_lps,
        name=f'Baseline (μ={mean_baseline:.2f})',
        xbins=dict(start=-12.0, end=0.0, size=0.4),
        marker_color='rgba(239, 68, 68, 0.65)', # Red
        opacity=0.75,
        hovertext=[f"Q: {q}" for q in questions]
    ))

    fig.add_trace(go.Histogram(
        x=ablated_lps,
        name=f'Ablated [Proj] (μ={mean_ablated:.2f})',
        xbins=dict(start=-12.0, end=0.0, size=0.4),
        marker_color='rgba(59, 130, 246, 0.65)', # Blue
        opacity=0.75,
        hovertext=[f"Q: {q}" for q in questions]
    ))
    
    fig.add_trace(go.Histogram(
        x=translated_lps,
        name=f'Translated [Inv] (μ={mean_translated:.2f})',
        xbins=dict(start=-12.0, end=0.0, size=0.4),
        marker_color='rgba(34, 197, 94, 0.65)', # Green
        opacity=0.75,
        hovertext=[f"Q: {q}" for q in questions]
    ))

    fig.update_layout(
        title_text=f"{model_name} Knowledge Recovery: Ablation vs. Translation",
        xaxis_title_text="Mean Token Log-Probability (Model Confidence)",
        yaxis_title_text="Number of Questions",
        barmode='overlay',
        template='plotly_white',
        width=950,
        height=550,
        legend=dict(yanchor="top", y=0.99, xanchor="left", x=0.01)
    )

    return fig

Experimental Loop:
- Isolate Global Mean-Shift Vector: Calculate the uniform translation vector for each unlearnin gmethod
- Normalize mean-shift vector
- Compare token probability for unlearned vs ablated unlearned model
- Plot


In [14]:
import matplotlib.pyplot as plt
import numpy as np
import gc
import torch
from tqdm import tqdm

NUM_LAYERS = 16
layer_names = [f"model.layers.{i}" for i in range(NUM_LAYERS)]

print("--- Pre-computing Base Model Activations ---")
base_model, base_tokenizer, device = load_model(BASE_MODEL)

# Dictionary to hold the base activations in CPU RAM
base_acts_dict = {}

for layer_name in tqdm(layer_names, desc="Extracting Base Layers"):
    acts = extract_activations(base_model, base_tokenizer, questions, layer_name, device)
    # acts is already a NumPy array, store directly
    base_acts_dict[layer_name] = acts

# Free the Base Model completely
del base_model
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
elif torch.backends.mps.is_available():
    torch.mps.empty_cache()

# Store results for plotting: {model_name: {layer_name: recovery_delta}}
sweep_results = {model: {"ablation": [], "translation": []} for model in models_to_eval.keys()}

for test_model, model_path in tqdm(models_to_eval.items(), desc="Evaluating Models"):
    # FIX: Properly unpack tokenizer and device
    unlearned_model, tokenizer, device = load_model(model_path)

    for layer_idx, layer_name in enumerate(layer_names):
        print(f"\n--- Profiling {test_model} at {layer_name} ---")
        
        unlearned_acts_np = extract_activations(unlearned_model, tokenizer, questions, layer_name, device)
        
        base_acts_layer = torch.tensor(base_acts_dict[layer_name], dtype=unlearned_model.dtype, device=device)
        unlearned_acts_layer = torch.tensor(unlearned_acts_np, dtype=unlearned_model.dtype, device=device)
        
        # 2. Calculate layer-specific shifts (Pure PyTorch Math)
        # Translation shift (Base - Unlearned)
        diffs = base_acts_layer - unlearned_acts_layer
        shift_tensor = diffs.mean(dim=0) 
        
        # Ablation direction (Unlearned - Base)
        gate_diffs = unlearned_acts_layer - base_acts_layer
        gate_shift = gate_diffs.mean(dim=0)
        direction_tensor = gate_shift / torch.norm(gate_shift)

        # 3. Run the evaluation loop (Scout pass: 5 questions)
        ablated_lps = []
        translated_lps = []
        baseline_lps = []
        
        for q, a in zip(questions[:5], correct_answers[:5]):
            baseline_lps.append(get_target_logprob(unlearned_model, tokenizer, q, a, direction_tensor, shift_tensor, mode="baseline", layer_target=layer_name))
            ablated_lps.append(get_target_logprob(unlearned_model, tokenizer, q, a, direction_tensor, shift_tensor, mode="ablate", layer_target=layer_name))
            translated_lps.append(get_target_logprob(unlearned_model, tokenizer, q, a, direction_tensor, shift_tensor, mode="translate", layer_target=layer_name))
            
        # Calculate Delta (Recovery - Baseline)
        ablation_delta = np.mean(ablated_lps) - np.mean(baseline_lps)
        translation_delta = np.mean(translated_lps) - np.mean(baseline_lps)
        
        sweep_results[test_model]["ablation"].append(ablation_delta)
        sweep_results[test_model]["translation"].append(translation_delta)
        
        # FIX: Correct memory cleanup names
        del unlearned_acts_np
        del unlearned_acts_layer
        del base_acts_layer
        torch.cuda.empty_cache()
        
    del unlearned_model
    torch.cuda.empty_cache()

--- Pre-computing Base Model Activations ---


Loading weights:   0%|          | 0/146 [00:00<?, ?it/s]

Loaded: open-unlearning/tofu_Llama-3.2-1B-Instruct_full
Device: mps | dtype: torch.float16
Params: 1.2B


Evaluating Models:   0%|          | 0/6 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/146 [00:00<?, ?it/s]

...

Loaded: open-unlearning/unlearn_tofu_Llama-3.2-1B-Instruct_forget10_SimNPO_lr2e-05_b4.5_a1_d1_g0.125_ep10
Device: mps | dtype: torch.float16
Params: 1.2B

--- Profiling SimNPO at model.layers.0 ---

--- Profiling SimNPO at model.layers.1 ---

--- Profiling SimNPO at model.layers.2 ---

--- Profiling SimNPO at model.layers.3 ---

--- Profiling SimNPO at model.layers.4 ---

--- Profiling SimNPO at model.layers.5 ---

--- Profiling SimNPO at model.layers.6 ---

--- Profiling SimNPO at model.layers.7 ---

--- Profiling SimNPO at model.layers.8 ---

--- Profiling SimNPO at model.layers.9 ---

--- Profiling SimNPO at model.layers.10 ---

--- Profiling SimNPO at model.layers.11 ---

--- Profiling SimNPO at model.layers.12 ---

--- Profiling SimNPO at model.layers.13 ---

--- Profiling SimNPO at model.layers.14 ---

--- Profiling SimNPO at model.layers.15 ---


Evaluating Models:  83%|████████▎ | 5/6 [19:44<03:55, 235.41s/it]

Loading weights:   0%|          | 0/146 [00:00<?, ?it/s]

Loaded: open-unlearning/unlearn_tofu_Llama-3.2-1B-Instruct_forget10_RMU_lr5e-05_layer10_scoeff10_epoch10
Device: mps | dtype: torch.float16
Params: 1.2B

--- Profiling RMU at model.layers.0 ---

--- Profiling RMU at model.layers.1 ---

--- Profiling RMU at model.layers.2 ---

--- Profiling RMU at model.layers.3 ---

--- Profiling RMU at model.layers.4 ---

--- Profiling RMU at model.layers.5 ---

--- Profiling RMU at model.layers.6 ---

--- Profiling RMU at model.layers.7 ---

--- Profiling RMU at model.layers.8 ---

--- Profiling RMU at model.layers.9 ---

--- Profiling RMU at model.layers.10 ---

--- Profiling RMU at model.layers.11 ---

--- Profiling RMU at model.layers.12 ---

--- Profiling RMU at model.layers.13 ---

--- Profiling RMU at model.layers.14 ---

--- Profiling RMU at model.layers.15 ---


Evaluating Models: 100%|██████████| 6/6 [23:37<00:00, 236.25s/it]


In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import gc

NUM_LAYERS = 16
layer_names = [f"model.layers.{i}" for i in range(NUM_LAYERS)]

print("--- Pre-computing Base Model Activations ---")
base_model, base_tokenizer, device = load_model(BASE_MODEL)

# Dictionary to hold the base activations in CPU RAM
base_acts_dict = {}

for layer_name in tqdm(layer_names, desc="Extracting Base Layers"):
    acts = extract_activations(base_model, base_tokenizer, questions, layer_name, device)
    # Move to CPU immediately so we don't hog GPU VRAM
    base_acts_dict[layer_name] = acts

# Free the Base Model completely
del base_model
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
elif torch.backends.mps.is_available():
    torch.mps.empty_cache()

# Store results for plotting: {model_name: {layer_name: recovery_delta}}
sweep_results = {model: {"ablation": [], "translation": []} for model in models_to_eval.keys()}

for test_model, model_path in tqdm(models_to_eval.items(), desc="Evaluating Models"):
    unlearned_model, _, _ = load_model(model_path)

    for layer_idx, layer_name in enumerate(layer_names):
        print(f"\n--- Profiling {test_model} at {layer_name} ---")
        
        unlearned_acts_np = extract_activations(unlearned_model, tokenizer, questions, layer_name, device)
        
        base_acts_layer = torch.tensor(base_acts_dict[layer_name], dtype=unlearned_model.dtype, device=device)
        unlearned_acts_layer = torch.tensor(unlearned_acts_np, dtype=unlearned_model.dtype, device=device)
        
        # 2. Calculate layer-specific shifts (Pure PyTorch Math)
        # Translation shift (Base - Unlearned)
        diffs = base_acts_layer - unlearned_acts_layer
        shift_tensor = diffs.mean(dim=0) # dim=0 instead of axis=0 for PyTorch
        
        # Ablation direction (Unlearned - Base)
        gate_diffs = unlearned_acts_layer - base_acts_layer
        gate_shift = gate_diffs.mean(dim=0)
        direction_tensor = gate_shift / torch.norm(gate_shift)

        # 3. Run the evaluation loop (you can sample just 100 questions here to save time)
        ablated_lps = []
        translated_lps = []
        baseline_lps = []
        
        for q, a in zip(questions[:5], correct_answers[:5]):
            baseline_lps.append(get_target_logprob(unlearned_model, tokenizer, q, a, direction_tensor, shift_tensor, mode="baseline", layer_target=layer_name))
            ablated_lps.append(get_target_logprob(unlearned_model, tokenizer, q, a, direction_tensor, shift_tensor, mode="ablate", layer_target=layer_name))
            translated_lps.append(get_target_logprob(unlearned_model, tokenizer, q, a, direction_tensor, shift_tensor, mode="translate", layer_target=layer_name))
            
        # Calculate Delta (Recovery - Baseline)
        ablation_delta = np.mean(ablated_lps) - np.mean(baseline_lps)
        translation_delta = np.mean(translated_lps) - np.mean(baseline_lps)
        
        sweep_results[test_model]["ablation"].append(ablation_delta)
        sweep_results[test_model]["translation"].append(translation_delta)
        
        
        # Memory cleanup per layer
        del unlearned_acts_np
        del unlearned_acts_layer
        del base_acts_layer
        torch.cuda.empty_cache()
        
    del unlearned_model
    torch.cuda.empty_cache()

--- Pre-computing Base Model Activations ---


Loading weights:   0%|          | 0/146 [00:00<?, ?it/s]

Loaded: open-unlearning/tofu_Llama-3.2-1B-Instruct_full
Device: mps | dtype: torch.float16
Params: 1.2B


Evaluating Models:   0%|          | 0/6 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/146 [00:00<?, ?it/s]

Loaded: open-unlearning/tofu_Llama-3.2-1B-Instruct_retain90
Device: mps | dtype: torch.float16
Params: 1.2B

--- Profiling Retain Oracle at model.layers.0 ---


/var/folders/1l/rwxq81q945x_8d2xvj7r5s4w0000gn/T/ipykernel_53116/1553574230.py:42: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  shift_tensor = torch.tensor(diffs.mean(axis=0), dtype=unlearned_model.dtype, device=device)
Evaluating Models:   0%|          | 0/6 [00:16<?, ?it/s]


TypeError: unsupported operand type(s) for -: 'numpy.ndarray' and 'Tensor'

In [ ]:
import json
import plotly.graph_objects as go

# 1. Save the Raw Data
save_path = os.path.join(OUTPUT_DIR, "layer_sweep_results.json")
with open(save_path, "w") as f:
    json.dump(sweep_results, f, indent=2)
print(f"Saved raw sweep data to {save_path}")

# 2. Generate and Save the Causal Trace Figures
for model_name, metrics in sweep_results.items():
    fig = go.Figure()

    # Add Ablation Line
    fig.add_trace(go.Scatter(
        x=list(range(NUM_LAYERS)), 
        y=metrics["ablation"], 
        mode='lines+markers',
        name='Ablation (Projection)',
        line=dict(color='blue', width=3),
        marker=dict(size=8)
    ))

    # Add Translation Line
    fig.add_trace(go.Scatter(
        x=list(range(NUM_LAYERS)), 
        y=metrics["translation"], 
        mode='lines+markers',
        name='Translation (Inversion)',
        line=dict(color='green', width=3),
        marker=dict(size=8)
    ))

    fig.update_layout(
        title=f"Causal Trace: {model_name} Knowledge Recovery by Layer",
        xaxis_title="Transformer Layer Index",
        yaxis_title="Recovery Delta (Δ Mean Log-Prob)",
        xaxis=dict(tickmode='linear', tick0=0, dtick=1), # Show every layer number
        template='plotly_white',
        width=900,
        height=500
    )

    # Save to HTML
    safe_name = model_name.replace(" ", "_")
    html_path = os.path.join(OUTPUT_DIR, f"{safe_name}_layer_trace.html")
    fig.write_html(html_path)
    print(f"Saved figure: {html_path}")